# Bases de datos · SQL avanzado

Continuación de *SQLite y SQL* con lo que va más allá del `SELECT … GROUP BY`: funciones de ventana, consultas con nombre (`WITH`) y recursivas, planes de consulta e índices compuestos, transacciones con puntos de guardado y aislamiento entre conexiones, restricciones con borrado en cascada, `UPSERT`, vistas y disparadores para auditoría, y JSON dentro de SQLite. Todo con el módulo estándar `sqlite3` sobre una base de datos de ventas de una cadena de tiendas de las islas (`sql_avanzado.db`, que crea el propio notebook) y cada consulta se comprueba repitiendo el cálculo con pandas o con Python.

## 1. Preparar los datos

Comprobar que la versión de SQLite que trae Python admite funciones de ventana (desde la 3.25) y el operador JSON `->>` (desde la 3.38). Crear las tablas `vendedores` (con su isla y su jefe directo) y `ventas` (vendedor, fecha e importe en euros), rellenar 500 ventas aleatorias del primer semestre de 2026 con semilla fija y cargar las dos tablas también en `DataFrame` para las comprobaciones.

In [1]:
import json
import random
import sqlite3
from collections import Counter, deque
from datetime import date, timedelta
from pathlib import Path

import pandas as pd

print("SQLite", sqlite3.sqlite_version)
assert sqlite3.sqlite_version_info >= (3, 38)

for nombre in ("sql_avanzado.db", "sql_avanzado.db-wal", "sql_avanzado.db-shm"):
    Path(nombre).unlink(missing_ok=True)
con = sqlite3.connect("sql_avanzado.db")
con.execute("PRAGMA foreign_keys = ON")
con.executescript('''
CREATE TABLE vendedores (
    id      INTEGER PRIMARY KEY,
    nombre  TEXT NOT NULL UNIQUE,
    isla    TEXT NOT NULL,
    jefe_id INTEGER REFERENCES vendedores (id)
);
CREATE TABLE ventas (
    id          INTEGER PRIMARY KEY,
    vendedor_id INTEGER NOT NULL REFERENCES vendedores (id),
    fecha       TEXT NOT NULL,
    importe     INTEGER NOT NULL CHECK (importe > 0)
);
''')
plantilla = [
    (1, "Nayra", "Gran Canaria", None), (2, "Airam", "Tenerife", 1), (3, "Tibisay", "Gran Canaria", 1),
    (4, "Echedey", "Tenerife", 2), (5, "Yaiza", "Lanzarote", 1), (6, "Acorán", "Tenerife", 2),
    (7, "Guayarmina", "Gran Canaria", 3), (8, "Tanausú", "Lanzarote", 5), (9, "Chaxiraxi", "Tenerife", 4),
    (10, "Doramas", "Gran Canaria", 3),
]
rng = random.Random(2026)
inicio = date(2026, 1, 1)
filas = []
for i in range(1, 501):
    vendedor = rng.choice([1, 2, 3, 4, 4, 5, 6, 6, 7, 7, 8, 9, 10, 10])
    dia = inicio + timedelta(days=rng.randrange(181))
    filas.append((i, vendedor, dia.isoformat(), rng.randint(20, 900)))
with con:
    con.executemany("INSERT INTO vendedores VALUES (?, ?, ?, ?)", plantilla)
    con.executemany("INSERT INTO ventas VALUES (?, ?, ?, ?)", filas)


def tabla(sql, params=()):
    return pd.read_sql_query(sql, con, params=params)


df_vendedores = tabla("SELECT * FROM vendedores")
df_ventas = tabla("SELECT * FROM ventas").merge(
    df_vendedores[["id", "nombre", "isla"]].rename(columns={"id": "vendedor_id"}), on="vendedor_id"
)
print(len(df_vendedores), "vendedores y", len(df_ventas), "ventas de", df_ventas.fecha.min(), "a", df_ventas.fecha.max())
display(df_ventas.head())
assert len(df_ventas) == 500 and df_ventas.importe.sum() == sum(f[3] for f in filas)

SQLite 3.50.4
10 vendedores y 500 ventas de 2026-01-01 a 2026-06-30


,id,vendedor_id,fecha,importe,nombre,isla
0,1,2,2026-03-23,534,Airam,Tenerife
1,2,7,2026-06-15,900,Guayarmina,Gran Canaria
2,3,2,2026-02-27,635,Airam,Tenerife
3,4,7,2026-05-23,450,Guayarmina,Gran Canaria
4,5,10,2026-05-27,580,Doramas,Gran Canaria


## 2. `ROW_NUMBER`, `RANK` y `DENSE_RANK`

En el concurso de parrandas de las fiestas hay empates. Numerar a los grupos por puntos de tres maneras: `ROW_NUMBER` (1, 2, 3… sin repetir, desempatando por nombre), `RANK` (los empatados comparten puesto y se salta los siguientes: 1, 2, 2, 4) y `DENSE_RANK` (sin saltos: 1, 2, 2, 3). Después, con `ROW_NUMBER() OVER (PARTITION BY isla …)`, obtener las dos ventas más grandes de cada isla. Comprobarlo con `rank(method=...)` y `groupby(...).head(2)` de pandas.

In [2]:
con.executescript('''
CREATE TABLE concurso (grupo TEXT PRIMARY KEY, puntos INTEGER NOT NULL);
INSERT INTO concurso VALUES
    ('Los Gofiones', 92), ('Parranda El Charco', 88), ('Taifas de Tejeda', 92), ('Los Sabandeños', 85),
    ('Barrio de Abajo', 88), ('Timples del Norte', 88), ('Rancho de Ánimas', 79);
''')
puestos = tabla('''
SELECT grupo, puntos,
       ROW_NUMBER() OVER (ORDER BY puntos DESC, grupo) AS fila,
       RANK()       OVER (ORDER BY puntos DESC)        AS rango,
       DENSE_RANK() OVER (ORDER BY puntos DESC)        AS denso
FROM concurso
ORDER BY fila
''')
display(puestos)

esperado = puestos[["grupo", "puntos"]].copy()
assert list(puestos.fila) == list(range(1, len(puestos) + 1))
assert list(puestos.grupo) == list(esperado.sort_values(["puntos", "grupo"], ascending=[False, True]).grupo)
assert list(puestos.rango) == list(esperado.puntos.rank(method="min", ascending=False).astype(int))
assert list(puestos.denso) == list(esperado.puntos.rank(method="dense", ascending=False).astype(int))

top = tabla('''
SELECT isla, id, nombre, importe
FROM (
    SELECT v.isla, s.id, v.nombre, s.importe,
           ROW_NUMBER() OVER (PARTITION BY v.isla ORDER BY s.importe DESC, s.id) AS n
    FROM ventas s JOIN vendedores v ON v.id = s.vendedor_id
)
WHERE n <= 2
ORDER BY isla, n
''')
display(top)
con_pandas = (df_ventas.sort_values(["isla", "importe", "id"], ascending=[True, False, True])
              .groupby("isla").head(2))
assert list(top.id) == list(con_pandas.id)

,grupo,puntos,fila,rango,denso
0,Los Gofiones,92,1,1,1
1,Taifas de Tejeda,92,2,1,1
2,Barrio de Abajo,88,3,3,2
3,Parranda El Charco,88,4,3,2
4,Timples del Norte,88,5,3,2
5,Los Sabandeños,85,6,6,3
6,Rancho de Ánimas,79,7,7,4


,isla,id,nombre,importe
0,Gran Canaria,2,Guayarmina,900
1,Gran Canaria,135,Guayarmina,898
2,Lanzarote,235,Yaiza,897
3,Lanzarote,208,Tanausú,887
4,Tenerife,482,Acorán,900
5,Tenerife,350,Acorán,896


## 3. `LAG` y `LEAD`: comparar con el mes anterior

Calcular el total de ventas de cada isla por mes y, en la misma fila, el del mes anterior (`LAG`), el del siguiente (`LEAD`) y la variación en porcentaje. Para no repetir la ventana tres veces, definirla una sola vez con `WINDOW w AS (PARTITION BY isla ORDER BY mes)`. Comprobarlo con `groupby(...).shift()`.

In [3]:
mensual = tabla('''
WITH totales AS (
    SELECT v.isla, strftime('%m', s.fecha) AS mes, SUM(s.importe) AS total
    FROM ventas s JOIN vendedores v ON v.id = s.vendedor_id
    GROUP BY v.isla, mes
)
SELECT isla, mes, total,
       LAG(total)  OVER w AS anterior,
       LEAD(total) OVER w AS siguiente,
       ROUND(100.0 * (total - LAG(total) OVER w) / LAG(total) OVER w, 1) AS variacion
FROM totales
WINDOW w AS (PARTITION BY isla ORDER BY mes)
ORDER BY isla, mes
''')
display(mensual[mensual.isla == "Tenerife"])

df_mes = (df_ventas.assign(mes=df_ventas.fecha.str[5:7]).groupby(["isla", "mes"], as_index=False).importe.sum()
          .rename(columns={"importe": "total"}))
grupos = df_mes.groupby("isla").total
pd.testing.assert_series_equal(mensual.anterior, grupos.shift(1), check_names=False, check_dtype=False)
pd.testing.assert_series_equal(mensual.siguiente, grupos.shift(-1), check_names=False, check_dtype=False)
pd.testing.assert_series_equal(mensual.variacion, grupos.pct_change() * 100, check_names=False, atol=0.051)

,isla,mes,total,anterior,siguiente,variacion
12,Tenerife,01,10947,NaN,15673.0,NaN
13,Tenerife,02,15673,10947.0,14817.0,43.2
14,Tenerife,03,14817,15673.0,11910.0,-5.5
15,Tenerife,04,11910,14817.0,23215.0,-19.6
16,Tenerife,05,23215,11910.0,16756.0,94.9
17,Tenerife,06,16756,23215.0,NaN,-27.8


## 4. Sumas acumuladas y medias móviles

Con las ventas diarias de Tenerife, calcular en una sola consulta:
- la suma acumulada desde el principio (`ROWS UNBOUNDED PRECEDING`),
- la media móvil de los tres últimos días con ventas (`ROWS BETWEEN 2 PRECEDING AND CURRENT ROW`),
- la suma acumulada que vuelve a empezar cada mes (`PARTITION BY` el mes).

Comprobarlo con `cumsum` y `rolling`. Después, sobre las ventas sueltas (con varias el mismo día), comparar `ROWS` con el marco por defecto `RANGE`, que trata como un bloque las filas empatadas en el `ORDER BY`.

In [4]:
diario = tabla('''
WITH dias AS (
    SELECT s.fecha, SUM(s.importe) AS total
    FROM ventas s JOIN vendedores v ON v.id = s.vendedor_id
    WHERE v.isla = 'Tenerife'
    GROUP BY s.fecha
)
SELECT fecha, total,
       SUM(total) OVER (ORDER BY fecha ROWS UNBOUNDED PRECEDING) AS acumulado,
       ROUND(AVG(total) OVER (ORDER BY fecha ROWS BETWEEN 2 PRECEDING AND CURRENT ROW), 2) AS media_3,
       SUM(total) OVER (PARTITION BY strftime('%m', fecha) ORDER BY fecha) AS acumulado_mes
FROM dias
ORDER BY fecha
''')
display(diario.head(6))
assert list(diario.acumulado) == list(diario.total.cumsum())
assert list(diario.media_3) == list(diario.total.rolling(3, min_periods=1).mean().round(2))
assert list(diario.acumulado_mes) == list(diario.groupby(diario.fecha.str[:7]).total.cumsum())

marcos = tabla('''
SELECT id, fecha, importe,
       SUM(importe) OVER (ORDER BY fecha ROWS UNBOUNDED PRECEDING) AS por_filas,
       SUM(importe) OVER (ORDER BY fecha) AS por_rango
FROM ventas
ORDER BY fecha, id
''')
distintas = marcos[marcos.por_filas != marcos.por_rango]
print(f"Filas en que ROWS y RANGE difieren: {len(distintas)} de {len(marcos)}")
display(marcos[marcos.fecha == distintas.fecha.iloc[0]])
assert marcos.por_filas.iloc[-1] == marcos.por_rango.iloc[-1] == df_ventas.importe.sum()
assert (marcos.por_rango == marcos.groupby("fecha").por_filas.transform("max")).all()

,fecha,total,acumulado,media_3,acumulado_mes
0,2026-01-01,648,648,648.00,648
1,2026-01-02,695,1343,671.50,1343
2,2026-01-03,53,1396,465.33,1396
3,2026-01-04,114,1510,287.33,1510
4,2026-01-06,811,2321,326.00,2321
5,2026-01-08,297,2618,407.33,2618


Filas en que ROWS y RANGE difieren: 330 de 500


,id,fecha,importe,por_filas,por_rango
1,65,2026-01-02,340,988,1683
2,187,2026-01-02,382,1370,1683
3,427,2026-01-02,313,1683,1683


## 5. Consultas encadenadas con `WITH`

Una cláusula `WITH` da nombre a subconsultas que se usan después como tablas. Encontrar a los vendedores que venden por encima de la media de su isla encadenando tres pasos: el total de cada vendedor, la media de esos totales por isla y la comparación. Comprobarlo con pandas.

In [5]:
encima = tabla('''
WITH por_vendedor AS (
    SELECT v.nombre, v.isla, SUM(s.importe) AS total
    FROM vendedores v JOIN ventas s ON s.vendedor_id = v.id
    GROUP BY v.id
),
por_isla AS (
    SELECT isla, AVG(total) AS media FROM por_vendedor GROUP BY isla
)
SELECT p.isla, p.nombre, p.total, ROUND(i.media, 1) AS media_isla
FROM por_vendedor p JOIN por_isla i USING (isla)
WHERE p.total > i.media
ORDER BY p.isla, p.total DESC
''')
display(encima)

por_vendedor = df_ventas.groupby(["isla", "nombre"], as_index=False).importe.sum()
por_vendedor["media"] = por_vendedor.groupby("isla").importe.transform("mean")
esperado = por_vendedor[por_vendedor.importe > por_vendedor.media]
assert sorted(encima.nombre) == sorted(esperado.nombre)

,isla,nombre,total,media_isla
0,Gran Canaria,Guayarmina,35396,25222.3
1,Gran Canaria,Doramas,32275,25222.3
2,Lanzarote,Yaiza,19867,19088.0
3,Tenerife,Acorán,34150,23329.5
4,Tenerife,Echedey,30584,23329.5


## 6. CTE recursiva: jerarquía de la plantilla

Una CTE recursiva (`WITH RECURSIVE`) tiene una parte inicial y otra que se une a sí misma hasta que no añade filas. Recorrer la plantilla desde la jefa (`jefe_id IS NULL`) calculando el nivel de cada persona y su cadena de mando, y mostrarla con sangría. Después, obtener todo el equipo que depende de Airam, directa o indirectamente, con la facturación conjunta. Comprobarlo con un BFS en Python.

In [6]:
jerarquia = tabla('''
WITH RECURSIVE equipo (id, nombre, nivel, cadena) AS (
    SELECT id, nombre, 0, nombre FROM vendedores WHERE jefe_id IS NULL
    UNION ALL
    SELECT v.id, v.nombre, e.nivel + 1, e.cadena || ' > ' || v.nombre
    FROM vendedores v JOIN equipo e ON v.jefe_id = e.id
)
SELECT * FROM equipo ORDER BY cadena
''')
for fila in jerarquia.itertuples():
    print("    " * fila.nivel + fila.nombre)

equipo_airam = tabla('''
WITH RECURSIVE debajo (id) AS (
    SELECT id FROM vendedores WHERE nombre = 'Airam'
    UNION ALL
    SELECT v.id FROM vendedores v JOIN debajo d ON v.jefe_id = d.id
)
SELECT v.nombre, COALESCE(SUM(s.importe), 0) AS total
FROM debajo d JOIN vendedores v ON v.id = d.id LEFT JOIN ventas s ON s.vendedor_id = v.id
GROUP BY v.id
ORDER BY v.nombre
''')
display(equipo_airam)

hijos = {}
for id_, nombre, isla, jefe in plantilla:
    hijos.setdefault(jefe, []).append(id_)
nombres = {id_: nombre for id_, nombre, *_ in plantilla}
nivel, cola = {1: 0}, deque([1])
while cola:
    v = cola.popleft()
    for h in hijos.get(v, []):
        nivel[h] = nivel[v] + 1
        cola.append(h)
assert {nombres[i]: n for i, n in nivel.items()} == dict(zip(jerarquia.nombre, jerarquia.nivel))

debajo, cola = set(), deque([2])
while cola:
    v = cola.popleft()
    debajo.add(v)
    cola.extend(hijos.get(v, []))
assert sorted(equipo_airam.nombre) == sorted(nombres[i] for i in debajo)
assert equipo_airam.total.sum() == df_ventas[df_ventas.vendedor_id.isin(debajo)].importe.sum()

Nayra
    Airam
        Acorán
        Echedey
            Chaxiraxi
    Tibisay
        Doramas
        Guayarmina
    Yaiza
        Tanausú


,nombre,total
0,Acorán,34150
1,Airam,16266
2,Chaxiraxi,12318
3,Echedey,30584


## 7. CTE recursiva: generar una serie de fechas

Un `GROUP BY fecha` no muestra los días sin ventas. Generar todos los días de marzo de 2026 con una CTE recursiva (`date(d, '+1 day')`), cruzarlos con `LEFT JOIN` con las ventas de Lanzarote y rellenar con 0 los huecos. Contar los días sin ventas y comprobarlo con `pd.date_range` y `reindex`.

In [7]:
marzo = tabla('''
WITH RECURSIVE dias (d) AS (
    SELECT '2026-03-01'
    UNION ALL
    SELECT date(d, '+1 day') FROM dias WHERE d < '2026-03-31'
),
lanzarote AS (
    SELECT s.fecha, s.importe
    FROM ventas s JOIN vendedores v ON v.id = s.vendedor_id
    WHERE v.isla = 'Lanzarote'
)
SELECT d AS fecha, COUNT(l.importe) AS ventas, COALESCE(SUM(l.importe), 0) AS total
FROM dias LEFT JOIN lanzarote l ON l.fecha = d
GROUP BY d
ORDER BY d
''')
print(f"{len(marzo)} días, {(marzo.ventas == 0).sum()} sin ventas en Lanzarote; total del mes: {marzo.total.sum()} €")
display(marzo.head(8))

fechas = pd.date_range("2026-03-01", "2026-03-31").strftime("%Y-%m-%d")
esperado = (df_ventas[df_ventas.isla == "Lanzarote"].groupby("fecha").importe.sum()
            .reindex(fechas, fill_value=0))
assert len(marzo) == 31 and list(marzo.fecha) == list(fechas)
assert list(marzo.total) == list(esperado)

31 días, 21 sin ventas en Lanzarote; total del mes: 7128 €


,fecha,ventas,total
0,2026-03-01,1,595
1,2026-03-02,0,0
2,2026-03-03,0,0
3,2026-03-04,0,0
4,2026-03-05,0,0
5,2026-03-06,2,1294
6,2026-03-07,0,0
7,2026-03-08,0,0


## 8. CTE recursiva: caminos en un grafo

Las conexiones entre islas (barco o avión) con su duración en minutos se guardan como aristas en una tabla, en los dos sentidos. Listar con una CTE recursiva todos los itinerarios sin repetir isla de Lanzarote a El Hierro, guardando el recorrido como texto y descartando las islas que ya están en él con `instr`. Quedarse con el más rápido y con el de menos escalas, y comprobarlo con una búsqueda en profundidad en Python.

In [8]:
conexiones = [
    ("Lanzarote", "Fuerteventura", 35), ("Lanzarote", "Gran Canaria", 50), ("Fuerteventura", "Gran Canaria", 45),
    ("Gran Canaria", "Tenerife", 80), ("Gran Canaria", "La Palma", 55), ("Gran Canaria", "El Hierro", 65),
    ("Tenerife", "La Gomera", 50), ("Tenerife", "La Palma", 40), ("Tenerife", "El Hierro", 45),
    ("La Gomera", "El Hierro", 90), ("La Gomera", "La Palma", 120),
]
with con:
    con.execute("CREATE TABLE conexiones (origen TEXT, destino TEXT, minutos INTEGER)")
    con.executemany("INSERT INTO conexiones VALUES (?, ?, ?)", conexiones + [(b, a, m) for a, b, m in conexiones])

caminos = tabla('''
WITH RECURSIVE ruta (isla, recorrido, minutos, escalas) AS (
    SELECT 'Lanzarote', ',Lanzarote,', 0, 0
    UNION ALL
    SELECT c.destino, r.recorrido || c.destino || ',', r.minutos + c.minutos, r.escalas + 1
    FROM ruta r JOIN conexiones c ON c.origen = r.isla
    WHERE r.isla <> 'El Hierro' AND instr(r.recorrido, ',' || c.destino || ',') = 0
)
SELECT trim(replace(recorrido, ',', ' > '), ' >') AS recorrido, minutos, escalas - 1 AS escalas
FROM ruta
WHERE isla = 'El Hierro'
ORDER BY minutos, escalas
''')
print(len(caminos), "itinerarios; los cinco más rápidos:")
display(caminos.head())
print("Con menos escalas:", caminos.sort_values(["escalas", "minutos"]).iloc[0].recorrido)

vecinos = {}
for a, b, m in conexiones:
    vecinos.setdefault(a, []).append((b, m))
    vecinos.setdefault(b, []).append((a, m))
todos = []
pila = [("Lanzarote", ["Lanzarote"], 0)]
while pila:
    isla, recorrido, minutos = pila.pop()
    if isla == "El Hierro":
        todos.append((" > ".join(recorrido), minutos))
        continue
    for siguiente, m in vecinos[isla]:
        if siguiente not in recorrido:
            pila.append((siguiente, recorrido + [siguiente], minutos + m))
assert sorted(todos) == sorted(zip(caminos.recorrido, caminos.minutos))
assert caminos.minutos.iloc[0] == min(m for _, m in todos)

16 itinerarios; los cinco más rápidos:


,recorrido,minutos,escalas
0,Lanzarote > Gran Canaria > El Hierro,115,1
1,Lanzarote > Fuerteventura > Gran Canaria > El ...,145,2
2,Lanzarote > Gran Canaria > Tenerife > El Hierro,175,2
3,Lanzarote > Gran Canaria > La Palma > Tenerife...,190,3
4,Lanzarote > Fuerteventura > Gran Canaria > Ten...,205,3


Con menos escalas: Lanzarote > Gran Canaria > El Hierro


## 9. Índices compuestos y `EXPLAIN QUERY PLAN`

Leer el plan de varias consultas sobre `ventas` antes y después de crear el índice compuesto `(vendedor_id, fecha)`:
- filtrar por vendedor y rango de fechas y ordenar por fecha: sin índice, `SCAN` y un árbol temporal para el `ORDER BY`; con índice, `SEARCH` y nada de ordenar, porque el índice ya está en ese orden;
- filtrar solo por fecha: el índice no sirve para buscar, porque su primera columna es el vendedor (como en una guía de teléfonos ordenada por apellido y nombre, no se puede buscar por nombre);
- pedir solo columnas del índice: `COVERING INDEX`, sin leer la tabla.

Comprobar que los resultados no cambian con el índice.

In [9]:
def plan(sql, params=()):
    return [fila[-1] for fila in con.execute("EXPLAIN QUERY PLAN " + sql, params)]


consultas = {
    "vendedor y fechas": ("SELECT fecha, importe FROM ventas WHERE vendedor_id = ? AND fecha >= ? ORDER BY fecha", (4, "2026-03-01")),
    "solo fecha": ("SELECT COUNT(*) FROM ventas WHERE fecha = ?", ("2026-03-02",)),
    "solo columnas del índice": ("SELECT fecha FROM ventas WHERE vendedor_id = ?", (9,)),
}
antes = {k: (plan(sql, p), con.execute(sql, p).fetchall()) for k, (sql, p) in consultas.items()}
con.execute("CREATE INDEX idx_vendedor_fecha ON ventas (vendedor_id, fecha)")
despues = {k: (plan(sql, p), con.execute(sql, p).fetchall()) for k, (sql, p) in consultas.items()}
for k in consultas:
    print(f"{k}:\n  sin índice: {antes[k][0]}\n  con índice: {despues[k][0]}")
    assert sorted(antes[k][1]) == sorted(despues[k][1])  # sin ORDER BY completo, el orden puede cambiar

assert any("TEMP B-TREE" in paso for paso in antes["vendedor y fechas"][0])
assert despues["vendedor y fechas"][0] == ["SEARCH ventas USING INDEX idx_vendedor_fecha (vendedor_id=? AND fecha>?)"]
assert all(paso.startswith("SCAN") for paso in despues["solo fecha"][0])
assert "COVERING INDEX" in despues["solo columnas del índice"][0][0]

con.execute("CREATE INDEX idx_fecha ON ventas (fecha)")
print("solo fecha, con índice propio:", plan(*consultas["solo fecha"]))
assert plan(*consultas["solo fecha"])[0].startswith("SEARCH")

vendedor y fechas:
  sin índice: ['SCAN ventas', 'USE TEMP B-TREE FOR ORDER BY']
  con índice: ['SEARCH ventas USING INDEX idx_vendedor_fecha (vendedor_id=? AND fecha>?)']
solo fecha:
  sin índice: ['SCAN ventas']
  con índice: ['SCAN ventas USING COVERING INDEX idx_vendedor_fecha']
solo columnas del índice:
  sin índice: ['SCAN ventas']
  con índice: ['SEARCH ventas USING COVERING INDEX idx_vendedor_fecha (vendedor_id=?)']
solo fecha, con índice propio: ['SEARCH ventas USING COVERING INDEX idx_fecha (fecha=?)']


## 10. Transacciones: `BEGIN`, `ROLLBACK` y `SAVEPOINT`

Con una conexión en modo `autocommit=True` (desde Python 3.12), las transacciones se controlan a mano con SQL. En una base de datos de cuentas de una cooperativa de plataneros:
- escribir `transferir(origen, destino, cantidad)` dentro de `BEGIN … COMMIT` y comprobar la atomicidad: si falla a mitad (el destino no existe, o el saldo quedaría negativo por la restricción `CHECK`), se hace `ROLLBACK` y no cambia nada;
- usar `SAVEPOINT` para deshacer solo una parte de una transacción con `ROLLBACK TO`.

In [10]:
banco = sqlite3.connect(":memory:", autocommit=True)
banco.executescript('''
CREATE TABLE cuentas (titular TEXT PRIMARY KEY, saldo INTEGER NOT NULL CHECK (saldo >= 0));
CREATE TABLE movimientos (id INTEGER PRIMARY KEY, concepto TEXT);
INSERT INTO cuentas VALUES ('Finca Los Llanos', 5000), ('Finca El Guincho', 1200), ('Cooperativa', 0);
''')


def saldos():
    return dict(banco.execute("SELECT titular, saldo FROM cuentas"))


def transferir(origen, destino, cantidad):
    banco.execute("BEGIN")
    try:
        banco.execute("UPDATE cuentas SET saldo = saldo - ? WHERE titular = ?", (cantidad, origen))
        if banco.execute("UPDATE cuentas SET saldo = saldo + ? WHERE titular = ?", (cantidad, destino)).rowcount != 1:
            raise ValueError(f"no existe la cuenta {destino!r}")
        banco.execute("COMMIT")
    except (ValueError, sqlite3.IntegrityError) as e:
        print(f"  {origen} -> {destino} ({cantidad} €) cancelada: {e}; en transacción: {banco.in_transaction}")
        banco.execute("ROLLBACK")


inicial = saldos()
transferir("Finca Los Llanos", "Cooperativa", 800)
transferir("Finca Los Llanos", "Cooperativa del Sur", 300)  # el primer UPDATE ya se había hecho
transferir("Finca El Guincho", "Cooperativa", 2000)
print("Saldos:", saldos())
assert saldos() == inicial | {"Finca Los Llanos": 4200, "Cooperativa": 800}
assert not banco.in_transaction

banco.execute("BEGIN")
banco.execute("INSERT INTO movimientos (concepto) VALUES ('cuota de enero')")
banco.execute("SAVEPOINT extra")
banco.execute("INSERT INTO movimientos (concepto) VALUES ('cargo duplicado')")
banco.execute("ROLLBACK TO extra")  # deshace desde el punto de guardado; la transacción sigue abierta
banco.execute("RELEASE extra")
banco.execute("INSERT INTO movimientos (concepto) VALUES ('cuota de febrero')")
banco.execute("COMMIT")
conceptos = [c for (c,) in banco.execute("SELECT concepto FROM movimientos ORDER BY id")]
print("Movimientos:", conceptos)
assert conceptos == ["cuota de enero", "cuota de febrero"]

  Finca Los Llanos -> Cooperativa del Sur (300 €) cancelada: no existe la cuenta 'Cooperativa del Sur'; en transacción: True
  Finca El Guincho -> Cooperativa (2000 €) cancelada: CHECK constraint failed: saldo >= 0; en transacción: True
Saldos: {'Finca Los Llanos': 4200, 'Finca El Guincho': 1200, 'Cooperativa': 800}
Movimientos: ['cuota de enero', 'cuota de febrero']


## 11. Aislamiento entre conexiones

Abrir dos conexiones a un mismo fichero en modo WAL (`PRAGMA journal_mode = WAL`) y comprobar el aislamiento de SQLite, que es serializable:
- lo que una conexión escribe dentro de una transacción no lo ve la otra hasta el `COMMIT`;
- una transacción de lectura trabaja sobre una instantánea: aunque otra conexión confirme cambios, no los ve hasta que termina;
- solo puede haber un escritor a la vez: un segundo `BEGIN IMMEDIATE` falla con `database is locked` (con `timeout=0` para no esperar).

In [11]:
for nombre in ("aislamiento.db", "aislamiento.db-wal", "aislamiento.db-shm"):
    Path(nombre).unlink(missing_ok=True)
escritor = sqlite3.connect("aislamiento.db", autocommit=True)
lector = sqlite3.connect("aislamiento.db", autocommit=True)
print("Modo:", escritor.execute("PRAGMA journal_mode = WAL").fetchone()[0])
escritor.execute("CREATE TABLE reservas (id INTEGER PRIMARY KEY, plaza TEXT)")


def cuenta(c):
    return c.execute("SELECT COUNT(*) FROM reservas").fetchone()[0]


escritor.execute("BEGIN")
escritor.execute("INSERT INTO reservas (plaza) VALUES ('1A')")
print("Sin confirmar  -> escritor ve", cuenta(escritor), "y lector ve", cuenta(lector))
assert (cuenta(escritor), cuenta(lector)) == (1, 0)
escritor.execute("COMMIT")
print("Confirmado     -> lector ve", cuenta(lector))
assert cuenta(lector) == 1

lector.execute("BEGIN")
foto = cuenta(lector)  # la instantánea se fija en la primera lectura
escritor.execute("INSERT INTO reservas (plaza) VALUES ('1B')")
print("Instantánea    -> lector ve", cuenta(lector), "aunque ya hay", cuenta(escritor))
assert cuenta(lector) == foto == 1
lector.execute("COMMIT")
print("Tras su COMMIT -> lector ve", cuenta(lector))
assert cuenta(lector) == 2

otro = sqlite3.connect("aislamiento.db", autocommit=True, timeout=0)
bloqueado = False
escritor.execute("BEGIN IMMEDIATE")
try:
    otro.execute("BEGIN IMMEDIATE")
except sqlite3.OperationalError as e:
    print("Segundo escritor:", e)
    bloqueado = True
escritor.execute("ROLLBACK")
otro.execute("BEGIN IMMEDIATE")
otro.execute("ROLLBACK")
assert bloqueado
for c in (escritor, lector, otro):
    c.close()

Modo: wal
Sin confirmar  -> escritor ve 1 y lector ve 0
Confirmado     -> lector ve 1
Instantánea    -> lector ve 1 aunque ya hay 2
Tras su COMMIT -> lector ve 2
Segundo escritor: database is locked


## 12. Restricciones y claves foráneas con `ON DELETE`

En una base de datos de islas, municipios y fiestas patronales:
- `NOT NULL`, `UNIQUE` (también compuesta: no puede haber dos municipios con el mismo nombre en una isla) y `CHECK` (población positiva);
- `municipios.isla_id` con `ON DELETE CASCADE` (al borrar la isla se borran sus municipios) y `fiestas.municipio_id` con `ON DELETE SET NULL`.

Comprobar que las claves foráneas están desactivadas por defecto en cada conexión, intentar insertar filas que violan cada regla mostrando el error y comprobar qué queda tras un borrado en cascada.

In [12]:
islas = sqlite3.connect(":memory:")
print("foreign_keys por defecto:", islas.execute("PRAGMA foreign_keys").fetchone()[0])
islas.execute("PRAGMA foreign_keys = ON")
islas.executescript('''
CREATE TABLE islas (id INTEGER PRIMARY KEY, nombre TEXT NOT NULL UNIQUE);
CREATE TABLE municipios (
    id        INTEGER PRIMARY KEY,
    isla_id   INTEGER NOT NULL REFERENCES islas (id) ON DELETE CASCADE,
    nombre    TEXT NOT NULL,
    poblacion INTEGER CHECK (poblacion > 0),
    UNIQUE (isla_id, nombre)
);
CREATE TABLE fiestas (
    id           INTEGER PRIMARY KEY,
    nombre       TEXT NOT NULL,
    municipio_id INTEGER REFERENCES municipios (id) ON DELETE SET NULL
);
INSERT INTO islas VALUES (1, 'La Palma'), (2, 'Fuerteventura');
INSERT INTO municipios VALUES (1, 1, 'Santa Cruz de La Palma', 15700), (2, 1, 'Los Llanos de Aridane', 20900),
                              (3, 2, 'Puerto del Rosario', 43000), (4, 2, 'Pájara', 21000);
INSERT INTO fiestas VALUES (1, 'Bajada de la Virgen', 1), (2, 'Los Indianos', 1), (3, 'Virgen de la Peña', NULL),
                           (4, 'San Miguel', 3);
''')
intentos = {
    "isla repetida": "INSERT INTO islas VALUES (3, 'La Palma')",
    "isla sin nombre": "INSERT INTO islas VALUES (3, NULL)",
    "municipio repetido en la isla": "INSERT INTO municipios VALUES (5, 1, 'Los Llanos de Aridane', 100)",
    "población negativa": "INSERT INTO municipios VALUES (5, 2, 'Betancuria', -800)",
    "isla inexistente": "INSERT INTO municipios VALUES (5, 9, 'Tijarafe', 2600)",
}
for motivo, sql in intentos.items():
    try:
        islas.execute(sql)
        print(f"{motivo:<30} aceptado (no debería)")
    except sqlite3.IntegrityError as e:
        print(f"{motivo:<30} {e}")
islas.execute("INSERT INTO municipios VALUES (5, 1, 'Pájara', 100)")  # nombre de prueba: repetido, pero en otra isla
assert islas.execute("SELECT COUNT(*) FROM municipios").fetchone()[0] == 5

islas.execute("DELETE FROM islas WHERE nombre = 'La Palma'")
print("Municipios:", islas.execute("SELECT nombre FROM municipios ORDER BY id").fetchall())
print("Fiestas:   ", islas.execute("SELECT nombre, municipio_id FROM fiestas ORDER BY id").fetchall())
assert islas.execute("SELECT COUNT(*) FROM municipios").fetchone()[0] == 2
assert islas.execute("SELECT nombre FROM fiestas WHERE municipio_id IS NULL ORDER BY id").fetchall() == [
    ("Bajada de la Virgen",), ("Los Indianos",), ("Virgen de la Peña",)]
islas.close()

foreign_keys por defecto: 0
isla repetida                  UNIQUE constraint failed: islas.nombre
isla sin nombre                NOT NULL constraint failed: islas.nombre
municipio repetido en la isla  UNIQUE constraint failed: municipios.isla_id, municipios.nombre
población negativa             CHECK constraint failed: poblacion > 0
isla inexistente               FOREIGN KEY constraint failed
Municipios: [('Puerto del Rosario',), ('Pájara',)]
Fiestas:    [('Bajada de la Virgen', None), ('Los Indianos', None), ('Virgen de la Peña', None), ('San Miguel', 3)]


## 13. `UPSERT` con `ON CONFLICT`

El almacén de una cooperativa recibe una lista de movimientos de cajas por producto. Con `INSERT … ON CONFLICT (producto) DO UPDATE SET unidades = unidades + excluded.unidades` (`excluded` es la fila que se intentaba insertar), sumar todos los movimientos en una tabla de existencias sin consultar antes si el producto existe, y comprobarlo con un `Counter`. Después, actualizar precios solo si la tarifa nueva es más reciente (`DO UPDATE … WHERE`), ignorar duplicados con `DO NOTHING` y ver lo que cambió con `RETURNING`.

In [13]:
almacen = sqlite3.connect(":memory:")
almacen.executescript('''
CREATE TABLE existencias (producto TEXT PRIMARY KEY, unidades INTEGER NOT NULL);
CREATE TABLE tarifas (producto TEXT PRIMARY KEY, precio REAL NOT NULL, desde TEXT NOT NULL);
''')
random.seed(13)
movimientos = [(random.choice(["plátano", "papaya", "aguacate", "mango", "tomate"]), random.randint(-5, 20))
               for _ in range(60)]
with almacen:
    almacen.executemany('''
        INSERT INTO existencias VALUES (?, ?)
        ON CONFLICT (producto) DO UPDATE SET unidades = unidades + excluded.unidades
    ''', movimientos)
existencias = dict(almacen.execute("SELECT producto, unidades FROM existencias ORDER BY producto"))
print("Existencias:", existencias)
esperado = Counter()
for producto, cantidad in movimientos:
    esperado[producto] += cantidad
assert existencias == dict(esperado)

with almacen:
    almacen.executemany("INSERT INTO tarifas VALUES (?, ?, ?)",
                        [("plátano", 0.95, "2026-01-01"), ("papaya", 1.80, "2026-03-01"), ("mango", 2.40, "2026-02-01")])
    cambios = almacen.execute('''
        INSERT INTO tarifas VALUES ('plátano', 1.05, '2026-04-01'), ('papaya', 1.60, '2026-02-01'),
                                   ('aguacate', 3.10, '2026-04-01')
        ON CONFLICT (producto) DO UPDATE SET precio = excluded.precio, desde = excluded.desde
            WHERE excluded.desde > tarifas.desde
        RETURNING producto, precio, desde
    ''').fetchall()
    ignorados = almacen.execute(
        "INSERT INTO tarifas VALUES ('mango', 9.99, '2026-05-01') ON CONFLICT DO NOTHING RETURNING producto"
    ).fetchall()
print("Cambiados:", cambios)
print("Ignorados con DO NOTHING devuelven:", ignorados)
tarifas = {p: (precio, desde) for p, precio, desde in almacen.execute("SELECT * FROM tarifas")}
assert sorted(cambios) == [("aguacate", 3.10, "2026-04-01"), ("plátano", 1.05, "2026-04-01")]
assert tarifas["papaya"] == (1.80, "2026-03-01") and tarifas["mango"] == (2.40, "2026-02-01") and ignorados == []

Existencias: {'aguacate': 102, 'mango': 97, 'papaya': 110, 'plátano': 64, 'tomate': 85}
Cambiados: [('plátano', 1.05, '2026-04-01'), ('aguacate', 3.1, '2026-04-01')]
Ignorados con DO NOTHING devuelven: []


## 14. Vistas y disparadores para auditoría

En una tabla de precios de una cadena de supermercados:
- disparadores `AFTER INSERT`, `AFTER UPDATE OF precio` y `AFTER DELETE` que anotan cada cambio en una tabla `auditoria` con los valores `OLD` y `NEW`;
- un disparador `BEFORE UPDATE` que rechaza con `RAISE(ABORT, …)` las subidas de más del 50 %;
- una vista `precios_con_igic` (con el 7 % de IGIC) y un disparador `INSTEAD OF UPDATE` que permite cambiar el precio final a través de la vista.

Repetir los mismos cambios sobre un diccionario en Python y comprobar que la auditoría coincide.

In [14]:
super_ = sqlite3.connect(":memory:")
super_.executescript('''
CREATE TABLE precios (producto TEXT PRIMARY KEY, precio REAL NOT NULL CHECK (precio > 0));
CREATE TABLE auditoria (id INTEGER PRIMARY KEY, accion TEXT, producto TEXT, antes REAL, despues REAL);

CREATE TRIGGER alta AFTER INSERT ON precios BEGIN
    INSERT INTO auditoria (accion, producto, despues) VALUES ('alta', NEW.producto, NEW.precio);
END;
CREATE TRIGGER cambio AFTER UPDATE OF precio ON precios WHEN NEW.precio <> OLD.precio BEGIN
    INSERT INTO auditoria (accion, producto, antes, despues) VALUES ('cambio', NEW.producto, OLD.precio, NEW.precio);
END;
CREATE TRIGGER baja AFTER DELETE ON precios BEGIN
    INSERT INTO auditoria (accion, producto, antes) VALUES ('baja', OLD.producto, OLD.precio);
END;
CREATE TRIGGER tope_subida BEFORE UPDATE OF precio ON precios WHEN NEW.precio > OLD.precio * 1.5 BEGIN
    SELECT RAISE(ABORT, 'subida de más del 50 %');
END;

CREATE VIEW precios_con_igic AS
    SELECT producto, precio, ROUND(precio * 1.07, 2) AS precio_final FROM precios;
CREATE TRIGGER cambiar_final INSTEAD OF UPDATE OF precio_final ON precios_con_igic BEGIN
    UPDATE precios SET precio = ROUND(NEW.precio_final / 1.07, 4) WHERE producto = NEW.producto;
END;
''')

operaciones = [
    ("INSERT INTO precios VALUES (?, ?)", ("queso majorero", 12.50)),
    ("INSERT INTO precios VALUES (?, ?)", ("mojo picón", 2.20)),
    ("INSERT INTO precios VALUES (?, ?)", ("bienmesabe", 4.80)),
    ("UPDATE precios SET precio = ? WHERE producto = ?", (13.10, "queso majorero")),
    ("UPDATE precios SET precio = ? WHERE producto = ?", (6.00, "mojo picón")),  # más del 50 %
    ("UPDATE precios SET precio = ? WHERE producto = ?", (4.80, "bienmesabe")),  # no cambia: no se audita
    ("UPDATE precios_con_igic SET precio_final = ? WHERE producto = ?", (2.68, "mojo picón")),
    ("DELETE FROM precios WHERE producto = ?", ("bienmesabe",)),
]
precios, esperado = {}, []
for sql, params in operaciones:
    try:
        with super_:
            super_.execute(sql, params)
    except sqlite3.IntegrityError as e:
        print("Rechazado:", params, "->", e)
    # la misma lógica en Python
    if sql.startswith("INSERT"):
        precios[params[0]] = params[1]
        esperado.append(("alta", params[0], None, params[1]))
    elif sql.startswith("DELETE"):
        esperado.append(("baja", params[0], precios.pop(params[0]), None))
    else:
        nuevo, producto = params
        if "igic" in sql:
            nuevo = round(nuevo / 1.07, 4)
        viejo = precios[producto]
        if nuevo > viejo * 1.5:
            continue
        if nuevo != viejo:
            precios[producto] = nuevo
            esperado.append(("cambio", producto, viejo, nuevo))

auditoria = super_.execute("SELECT accion, producto, antes, despues FROM auditoria ORDER BY id").fetchall()
for fila in auditoria:
    print(fila)
display(pd.read_sql_query("SELECT * FROM precios_con_igic", super_))
assert auditoria == esperado
assert dict(super_.execute("SELECT producto, precio FROM precios")) == precios

Rechazado: (6.0, 'mojo picón') -> subida de más del 50 %
('alta', 'queso majorero', None, 12.5)
('alta', 'mojo picón', None, 2.2)
('alta', 'bienmesabe', None, 4.8)
('cambio', 'queso majorero', 12.5, 13.1)
('cambio', 'mojo picón', 2.2, 2.5047)
('baja', 'bienmesabe', 4.8, None)


,producto,precio,precio_final
0,queso majorero,13.1000,14.02
1,mojo picón,2.5047,2.68


## 15. JSON dentro de SQLite

Los pedidos de la tienda en línea llegan como JSON y se guardan tal cual en una columna de texto. Con las funciones JSON de SQLite:
- sacar campos con `json_extract` y con el operador `->>`, y filtrar por ellos;
- desplegar la lista de artículos de cada pedido en filas con `json_each` y calcular el importe de cada pedido y las unidades por producto;
- construir un JSON de resumen por isla con `json_group_array` y `json_group_object`.

Comprobar cada resultado procesando los mismos pedidos con el módulo `json`.

In [15]:
random.seed(15)
catalogo = {"gofio": 2.5, "mojo": 3.0, "queso": 11.0, "vino": 9.5, "miel de palma": 7.0}
pedidos_json = []
for i in range(1, 9):
    articulos = [{"producto": p, "cantidad": random.randint(1, 4), "precio": catalogo[p]}
                 for p in random.sample(sorted(catalogo), random.randint(1, 3))]
    pedidos_json.append(json.dumps({
        "cliente": {"nombre": f"Cliente {i}", "isla": random.choice(["Tenerife", "Gran Canaria", "La Gomera"])},
        "urgente": random.random() < 0.4,
        "articulos": articulos,
    }, ensure_ascii=False))
with con:
    con.execute("CREATE TABLE pedidos_web (id INTEGER PRIMARY KEY, datos TEXT NOT NULL CHECK (json_valid(datos)))")
    con.executemany("INSERT INTO pedidos_web (datos) VALUES (?)", [(p,) for p in pedidos_json])

campos = tabla('''
SELECT id, json_extract(datos, '$.cliente.nombre') AS cliente, datos ->> '$.cliente.isla' AS isla,
       datos ->> '$.urgente' AS urgente, json_array_length(datos, '$.articulos') AS articulos
FROM pedidos_web
WHERE datos ->> '$.cliente.isla' <> 'La Gomera'
''')
display(campos)
pedidos = [json.loads(p) for p in pedidos_json]
assert list(campos.cliente) == [p["cliente"]["nombre"] for p in pedidos if p["cliente"]["isla"] != "La Gomera"]
assert list(campos.urgente) == [int(p["urgente"]) for p in pedidos if p["cliente"]["isla"] != "La Gomera"]

importes = tabla('''
SELECT p.id, ROUND(SUM((a.value ->> '$.cantidad') * (a.value ->> '$.precio')), 2) AS importe
FROM pedidos_web p, json_each(p.datos, '$.articulos') AS a
GROUP BY p.id
ORDER BY p.id
''')
unidades = dict(con.execute('''
SELECT a.value ->> '$.producto', SUM(a.value ->> '$.cantidad')
FROM pedidos_web p, json_each(p.datos, '$.articulos') AS a
GROUP BY 1
'''))
print("Importes:", list(importes.importe))
print("Unidades:", unidades)
assert list(importes.importe) == [round(sum(a["cantidad"] * a["precio"] for a in p["articulos"]), 2) for p in pedidos]
esperado = Counter()
for p in pedidos:
    for a in p["articulos"]:
        esperado[a["producto"]] += a["cantidad"]
assert unidades == dict(esperado)

resumen = json.loads(con.execute('''
SELECT json_group_object(isla, json(pedidos)) FROM (
    SELECT datos ->> '$.cliente.isla' AS isla, json_group_array(id) AS pedidos
    FROM pedidos_web GROUP BY isla
)
''').fetchone()[0])
print("Pedidos por isla:", resumen)
por_isla = {}
for i, p in enumerate(pedidos, 1):
    por_isla.setdefault(p["cliente"]["isla"], []).append(i)
assert resumen == por_isla
con.close()

,id,cliente,isla,urgente,articulos
0,1,Cliente 1,Tenerife,0,1
1,3,Cliente 3,Gran Canaria,1,3
2,4,Cliente 4,Tenerife,1,2
3,5,Cliente 5,Tenerife,0,3
4,6,Cliente 6,Tenerife,0,2
5,8,Cliente 8,Tenerife,1,3


Importes: [2.5, 2.5, 44.5, 49.5, 92.0, 56.0, 14.0, 27.0]
Unidades: {'gofio': 13, 'miel de palma': 9, 'mojo': 9, 'queso': 9, 'vino': 7}
Pedidos por isla: {'Gran Canaria': [3], 'La Gomera': [2, 7], 'Tenerife': [1, 4, 5, 6, 8]}
